# 02 — Exploratory Data Analysis

Reproduces proposal figures and adds academic-period and correlation views that inform feature choices.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if (ROOT / "rec_center_utils.py").exists():
    pass
elif (ROOT / "rec_center" / "rec_center_utils.py").exists():
    ROOT = ROOT / "rec_center"
elif (ROOT.parent / "rec_center_utils.py").exists():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("Could not locate rec_center_utils.py")
sys.path.insert(0, str(ROOT))

from rec_center_utils import (
    TRAIN_END,
    VAL_END,
    clean_data,
    data_path,
    figures_path,
    load_clean_data,
    load_raw_data,
    save_clean_data,
)

sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

df = load_clean_data()
df.shape



## Target distribution


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["average_utilization"], bins=50, kde=True, ax=ax)
ax.set_title("Distribution of Average Utilization")
fig.savefig(figures_path("average_utilization_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()



## Utilization by hour, weekday, and location


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=df.groupby("hour")["average_utilization"].mean().reset_index(), x="hour", y="average_utilization", ax=axes[0])
axes[0].set_title("By Hour")
sns.barplot(data=df.groupby("day_of_week")["average_utilization"].mean().reset_index(), x="day_of_week", y="average_utilization", ax=axes[1])
axes[1].set_title("By Weekday")
loc = df.groupby("location")["average_utilization"].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=loc, y="location", x="average_utilization", ax=axes[2])
axes[2].set_title("By Location")
fig.tight_layout()
for name, ax in zip(["utilization_by_hour.png", "utilization_by_weekday.png", "utilization_by_location.png"], axes):
    ax.figure.savefig(figures_path(name), dpi=150, bbox_inches="tight")
plt.show()



## Monthly trend and weekday-hour heatmap


In [ ]:

monthly = df.groupby(df["timestamp"].dt.to_period("M"))["average_utilization"].mean()
fig, ax = plt.subplots(figsize=(9, 4))
monthly.plot(ax=ax)
ax.set_title("Monthly Average Utilization Trend")
fig.savefig(figures_path("monthly_utilization_trend.png"), dpi=150, bbox_inches="tight")
plt.show()

pivot = df.pivot_table(values="average_utilization", index="day_of_week", columns="hour", aggfunc="mean")
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax)
ax.set_title("Weekday-Hour Heatmap")
fig.savefig(figures_path("weekday_hour_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()



## Additional insights for modeling


In [ ]:

period = pd.Series("In Quarter", index=df.index)
period[df["is_summer"] == 1] = "Summer"
period[df["is_winter_break"] == 1] = "Winter Break"
period_df = df.assign(academic_period=period)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=period_df.groupby("academic_period")["average_utilization"].mean().reset_index(),
    x="academic_period",
    y="average_utilization",
    order=["In Quarter", "Summer", "Winter Break"],
    ax=ax,
)
ax.set_title("Utilization by Academic Period")
fig.savefig(figures_path("utilization_by_academic_period.png"), dpi=150, bbox_inches="tight")
plt.show()

corr_cols = ["hour", "day_of_week", "month", "is_weekend", "is_summer", "is_finals_week", "average_utilization"]
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Feature Correlation Heatmap")
fig.savefig(figures_path("feature_correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()



### EDA takeaways

- Peak crowding occurs on weekday afternoons (especially 4–6 PM).
- Track and 2nd Floor areas run hotter than 1st Floor and Lower Exercise Room.
- Summer and winter break periods are materially quieter.
- Nonlinear time interactions justify tree/boosting models over a simple linear baseline.
